In [1]:
from utils.analysis.valuation import BuySellSignalsAnalyzer, SignalsReporter
from utils.data import DataManager
from utils.tools.config import ANALYSIS_START_DATE, ANALYSIS_END_DATE


In [2]:
# CONFIGURATION
# Dates come from config.py (TRADING_SIGNALS_CONFIG)
# Customize here only if you need different values

# Tickers to analyze
TICKERS  = ["IONQ", "MSFT", "GOOGL", "AMZN", "NVDA", "IBM"]   # Add more: ["AAPL", "MSFT", "GOOGL", ...]

# Lookback for price analysis
LOOKBACK_YEARS = 5  # Historical years

# (Optional) Custom dates - Leave empty to use config.py
USE_CUSTOM_DATES = False
CUSTOM_START = ""  # e.g.: "2022-01-01"
CUSTOM_END = ""    # e.g.: "2024-12-31"

print("Configuration:")
print(f"   Tickers: {len(TICKERS)}")
print(f"   Lookback: {LOOKBACK_YEARS} years")
if USE_CUSTOM_DATES and CUSTOM_START and CUSTOM_END:
    print(f"   Custom dates: {CUSTOM_START} -> {CUSTOM_END}")
else:
    print(f"   Dates from config.py: {ANALYSIS_START_DATE} -> {ANALYSIS_END_DATE}")

Configuration:
   Tickers: 6
   Lookback: 5 years
   Dates from config.py: 2020-01-01 -> 2025-12-24


In [3]:
# INITIALIZATION
# Create shared DataManager instance
data_manager = DataManager()

# Create analyzer with configuration
# If USE_CUSTOM_DATES is True, use custom dates
if USE_CUSTOM_DATES and CUSTOM_START and CUSTOM_END:
    analyzer = BuySellSignalsAnalyzer(
        data_manager=data_manager,
        start_date=CUSTOM_START,
        end_date=CUSTOM_END,
        lookback_years=LOOKBACK_YEARS
    )
    print(f"[OK] Analyzer with custom dates: {CUSTOM_START} -> {CUSTOM_END}")
else:
    analyzer = BuySellSignalsAnalyzer(
        data_manager=data_manager,
        lookback_years=LOOKBACK_YEARS
    )
    print(f"[OK] Analyzer with config.py settings")

reporter = SignalsReporter()
print("[OK] Reporter initialized")

[OK] Analyzer with config.py settings
[OK] Reporter initialized


In [4]:
# SIGNAL ANALYSIS
signals = []

print(f"\nAnalyzing {len(TICKERS)} ticker(s)...\n")

for ticker in TICKERS:
    try:
        # The analyzer already has the configured dates
        signal = analyzer.analyze_stock(ticker)
        signals.append(signal)
        
        # Show quick summary
        tag = "[BUY]" if signal.signal == "BUY" else "[SELL]" if signal.signal == "SELL" else "[HOLD]"
        print(f"{tag} {ticker}: {signal.signal} "
              f"(Conf: {signal.confidence:.1f}%, Pot: {signal.upside_potential:+.1f}%)")
        
    except Exception as e:
        print(f"[ERROR] {ticker}: {str(e)[:60]}")

print(f"\nTotal analyzed: {len(signals)}/{len(TICKERS)} tickers")


Analyzing 6 ticker(s)...

Period: 2020-01-01 → 2026-02-23


[*********************100%***********************]  1 of 1 completed


[HOLD] IONQ: HOLD (Conf: 50.0%, Pot: +0.8%)
Period: 2020-01-01 → 2026-02-23


[*********************100%***********************]  1 of 1 completed


[HOLD] MSFT: HOLD (Conf: 40.0%, Pot: +0.5%)
Period: 2020-01-01 → 2026-02-23


...

[HOLD] AMZN: HOLD (Conf: 50.0%, Pot: +0.3%)
Period: 2020-01-01 → 2026-02-23


[*********************100%***********************]  1 of 1 completed


[HOLD] NVDA: HOLD (Conf: 40.0%, Pot: +0.3%)
Period: 2020-01-01 → 2026-02-23


[*********************100%***********************]  1 of 1 completed

[HOLD] IBM: HOLD (Conf: 40.0%, Pot: +0.3%)

Total analyzed: 6/6 tickers


In [5]:
# SUMMARY TABLE
if signals:
    summary_df = reporter.to_dataframe(signals)
    display(summary_df)
    
    print()
    reporter.print_summary(signals)
else:
    print("[!] No signals to display")

,Ticker,Signal,Confidence,Valuation,Fundamental,Current Price,Target Price,Potential
0,IONQ,HOLD,50.0%,29.0,47.1,$32,$56,75.0%
1,MSFT,HOLD,40.0%,45.0,71.9,$397,$596,50.0%
2,GOOGL,HOLD,48.0%,28.8,68.5,$315,$377,19.6%
3,AMZN,HOLD,50.0%,27.0,61.4,$210,$281,33.5%
4,NVDA,HOLD,40.0%,24.2,85.1,$190,$254,33.8%
5,IBM,HOLD,40.0%,53.0,48.5,$257,$325,26.4%




SIGNALS SUMMARY

🟢 BUYS: 0

🔴 SELLS: 0

🟡 HOLDS: 6


In [6]:
# TOP BUY OPPORTUNITIES
if signals:
    reporter.print_top_opportunities(signals, top_n=5)
else:
    print("[!] No signals to analyze")


⚠️ No buy opportunities identified


In [7]:
# BUY SIGNAL DETAILS
buys = [s for s in signals if s.signal == "BUY"]

if buys:
    # Sort by confidence
    top_buys = sorted(buys, key=lambda x: x.confidence, reverse=True)
    
    print(f"Found {len(buys)} BUY signals\n")
    print("=" * 65)
    
    for signal in top_buys:
        reporter.print_signal(signal)
        print()
else:
    print("[INFO] No BUY signals at this time")

[INFO] No BUY signals at this time


In [8]:
# SECTOR ANALYSIS
# Analyze multiple tickers from a specific sector

TECH_SECTOR  = ["MSFT", "META", "NVDA", "GOOGL", "AMZN", "IBM"]

print("Analyzing technology sector...\n")
tech_signals = []

for ticker in TECH_SECTOR:
    try:
        signal = analyzer.analyze_stock(ticker)
        tech_signals.append(signal)
    except Exception as e:
        print(f"[!] {ticker}: {str(e)[:40]}")

if tech_signals:
    # Show only the best opportunities
    buys_tech = [s for s in tech_signals if s.signal == "BUY"]
    
    if buys_tech:
        print(f"\n[OK] {len(buys_tech)} buy opportunities in Tech:\n")
        
        tech_df = reporter.to_dataframe(buys_tech)
        display(tech_df.sort_values('Confidence', ascending=False))
    else:
        print("\n[INFO] No buy signals in the tech sector currently")

Analyzing technology sector...



[*********************100%***********************]  1 of 1 completed

Period: 2020-01-01 → 2026-02-23


Period: 2020-01-01 → 2026-02-23


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

...

Period: 2020-01-01 → 2026-02-23



[*********************100%***********************]  1 of 1 completed

Period: 2020-01-01 → 2026-02-23



[*********************100%***********************]  1 of 1 completed

Period: 2020-01-01 → 2026-02-23

[INFO] No buy signals in the tech sector currently
